# 09 · Inferencia y app de Streamlit

Muestra cómo se usa `models/model.joblib` **sin entrenar nada**: puntuar solicitudes, asignar la zona (verde / amarillo / rojo) y explicar una decisión con SHAP local. También genera `models/demo_requests.json` (un ejemplo por zona) que la app ofrece como atajo, y comprueba que la app arranca.

Para abrir la app: `streamlit run app/streamlit_app.py` desde la raíz del repositorio.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, joblib
import matplotlib; matplotlib.use("Agg")
from config.settings import PROCESSED_DIR, TARGET
from accessguard.explain import assign_zone, make_explainer, explain_request, unseen_codes, describe_feature, ZONE_LABELS

bundle = joblib.load(ROOT / "models" / "model.joblib")
pipe, T_AUTO, T_LOW, COLS = bundle["pipeline"], bundle["t_auto_strict"], bundle["t_low"], bundle["input_columns"]
print(f"{bundle['model_name']} · T_auto = {T_AUTO:.4f} · T_low = {T_LOW:.4f} · columnas de entrada: {COLS}")

LogReg ElasticNet · T_auto = 0.9809 · T_low = 0.7450 · columnas de entrada: ['RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC']


## 1. Puntuar un lote
El paquete recibe solo los códigos de la solicitud; el pipeline construye las variables internamente.

In [2]:
hold = pd.read_csv(PROCESSED_DIR / "holdout_clean.csv")        # solo como fuente de solicitudes de ejemplo
X = hold[COLS].sample(2000, random_state=42)
p = pipe.predict_proba(X)[:, 1]
zone = assign_zone(p, T_LOW, T_AUTO)
pd.Series(zone).value_counts().rename("solicitudes").to_frame().assign(decision=lambda d: d.index.map(ZONE_LABELS))

,solicitudes,decision
amarillo,1690,Cola de triaje priorizada
rojo,204,Revisión manual
verde,106,Auto-aprobación estricta


## 2. Ejemplos de demostración (uno por zona)
Se toma la solicitud cuya `p` es la más cercana a la mediana de cada zona. La elección usa solo la `p` del modelo, no las etiquetas.

In [3]:
pa = pipe.predict_proba(hold[COLS])[:, 1]
za = assign_zone(pa, T_LOW, T_AUTO)
demo = {}
for z, nombre in [("verde", "Ejemplo zona verde"), ("amarillo", "Ejemplo zona amarilla"), ("rojo", "Ejemplo zona roja")]:
    idx = np.flatnonzero(za == z)
    j = idx[np.argmin(np.abs(pa[idx] - np.median(pa[idx])))]
    demo[nombre] = {c: int(hold.iloc[j][c]) for c in COLS}
(ROOT / "models" / "demo_requests.json").write_text(json.dumps(demo, indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(demo).T

,RESOURCE,MGR_ID,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC
Ejemplo zona verde,31232,17487,117891,117878,118568,292195
Ejemplo zona amarilla,5173,5242,118052,120671,120357,120065
Ejemplo zona roja,38082,17841,118587,117941,118568,135898


## 3. Explicación local de una solicitud (SHAP)

In [4]:
explainer = make_explainer(pipe, bundle["shap_background"])
for nombre, row in demo.items():
    Xr = pd.DataFrame([row], columns=COLS).astype("int64")
    e = explain_request(pipe, explainer, Xr, top=4)
    z = assign_zone([e["p_approve"]], T_LOW, T_AUTO)[0]
    print(f"{nombre}: p = {e['p_approve']:.3f} → zona {z} · códigos nuevos: {unseen_codes(pipe, Xr) or 'ninguno'}")
    for t in e["top"]:
        print(f"    {t['shap']:+.2f}  {describe_feature(t['feature'])}")

Background dataset has 300 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=300 when initializing the masker.


Ejemplo zona verde: p = 0.987 → zona verde · códigos nuevos: ['MGR_ID']
    +1.34  Frecuencia de RESOURCE × ROLE_DEPTNAME
    -1.06  Frecuencia de RESOURCE × MGR_ID
    +0.58  Frecuencia de MGR_ID × ROLE_TITLE
    +0.21  Aprobación histórica de RESOURCE
Ejemplo zona amarilla: p = 0.888 → zona amarillo · códigos nuevos: ['MGR_ID']
    -1.06  Frecuencia de RESOURCE × MGR_ID
    +0.58  Frecuencia de MGR_ID × ROLE_TITLE
    -0.50  Frecuencia de RESOURCE × ROLE_DEPTNAME
    -0.30  Aprobación histórica de ROLE_ROLLUP_2
Ejemplo zona roja: p = 0.655 → zona rojo · códigos nuevos: ['RESOURCE', 'MGR_ID']
    -1.06  Frecuencia de RESOURCE × MGR_ID
    -1.06  Frecuencia de RESOURCE × ROLE_DEPTNAME
    +0.58  Frecuencia de MGR_ID × ROLE_TITLE
    -0.51  Aprobación histórica de ROLE_ROLLUP_2


## 4. Comprobación de la app
Se ejecuta la app en modo de prueba (sin navegador) y se verifica que carga el modelo, evalúa una solicitud y muestra la zona.

In [5]:
from streamlit.testing.v1 import AppTest
at = AppTest.from_file(str(ROOT / "app" / "streamlit_app.py"), default_timeout=120).run()
assert not at.exception, at.exception
for c, v in demo["Ejemplo zona roja"].items():
    at.number_input(key=f"in_{c}").set_value(v)
at.button[0].click().run()
assert not at.exception, at.exception
banner = [m.value for m in at.markdown if "ROJO" in m.value or "AMARILLO" in m.value or "VERDE" in m.value]
print("La app respondió:", banner[0][-120:] if banner else "sin banner")

2026-11-06 19:42:29.584 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Background dataset has 300 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=300 when initializing the masker.


La app respondió: 2rem;border-radius:6px'><div style='color:#A4262C;font-size:1.4rem;font-weight:700'>🔴 ROJO · Revisión manual</div></div>


## 5. Notas
* Un código que el modelo nunca vio aparece como aviso en la app; el modelo no tiene historial de él.
* La app **no** consulta Databricks: la fuente de verdad es `models/model.joblib`.
* El archivo `models/demo_requests.json` contiene solo códigos anónimos del dataset.